In [ ]:
# LIBRERIAS

from astropy.table import Table
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import pymc3 as pm
import pymc3_ext as pmx
import exoplanet as xo
import os
from corner import corner
import theano.tensor as tt
from tqdm.notebook import tqdm
from scipy.stats import median_abs_deviation
import pandas as pd
from astroquery.mast import Catalogs
from scipy.interpolate import UnivariateSpline
import pandas as pd

from pathlib import Path
from scipy.interpolate import interp1d

from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.ticker import ScalarFormatter

from scipy.interpolate import LinearNDInterpolator



# CATALOGOS

In [ ]:
# Catálogos

# Observables obtenidos de teff, spt con massage 

# Catalogo de resultados massage, el _1 indica modelo de mist y el _2 indica modelo de somers

path_cat = "/home/luciam/Documents/starry/Resultados_Orion/catalogo_completo_mist_sommers.csv"

cat = pd.read_csv(path_cat)
print(f"datos catálogo mist-somers:{cat.columns}")

#Catálogo de resultados obtenidos de la curva de luz con tess

path_priors = "/home/luciam/Documents/starry/Resultados_Orion/priors_completosv1.csv"

priors = pd.read_csv(path_priors)
print(f"datos catálogo priors:{priors.columns}")

#Malla de tracks de mist 

path_model_1 = "/home/luciam/Documents/starry/Resultados_Orion/datos_estrellas_massage_orion/modelos/MIST_V2.csv"

mist = pd.read_csv(path_model_1)
print(f"datos tracks mist:{mist.columns}")

# Malla de tracks Somers 17% manchas
path_model_2 = "/home/luciam/Documents/starry/Resultados_Orion/datos_estrellas_massage_orion/modelos/f017track_ms.csv"


somers17 = pd.read_csv(path_model_2)
print(f"datos tracks somers:{somers17.columns}")

In [ ]:
#Constantes del modelo

T_sun = 5772.0  # K


# identificadores en cada catálogo


# VARIABLES DEL MODELO

In [ ]:
# ============================================================
# DATOS OBSERVACIONALES — MIST
# ============================================================

data_obs_mist = cat[[
    "TIC",
    "T_mean_1",
    "T_std_1",
    "lgL_mean_1",
    "lgL_std_1",
    "age_mean_1",
    "age_std_1",
    "mass_mean_1",
    "mass_std_1"
]].copy()


# ============================================================
# DATOS OBSERVACIONALES — SOMERS+17
# ============================================================

data_obs_somers = cat[[
    "TIC",
    "T_mean_2",
    "T_std_2",
    "lgL_mean_2",
    "lgL_std_2",
    "age_mean_2",
    "age_std_2",
    "mass_mean_2",
    "mass_std_2"
]].copy()


# ============================================================
# AGREGAR DATOS DE TESS — MIST
# ============================================================

data_obs_mist = data_obs_mist.merge(
    priors[[
        "TIC",
        "Depth_primary",
        "Depth_primary_err"
    ]],
    on="TIC",
    how="inner"
)


# ============================================================
# AGREGAR DATOS DE TESS — SOMERS+17
# ============================================================

data_obs_somers = data_obs_somers.merge(
    priors[[
        "TIC",
        "Depth_primary",
        "Depth_primary_err"
    ]],
    on="TIC",
    how="inner"
)

# CALCULO DE RADIOS a partir de observables

In [ ]:
# ============================================================
# RADIO PRIMARIO — MIST
# ============================================================

data_obs_mist["R1_mean"] = (
    10**(data_obs_mist["lgL_mean_1"] / 2)
    * (T_sun / data_obs_mist["T_mean_1"])**2
)

data_obs_mist["R1_std"] = (
    data_obs_mist["R1_mean"]
    * np.sqrt(
        ((np.log(10) / 2) * data_obs_mist["lgL_std_1"])**2
        +
        (2 * data_obs_mist["T_std_1"]
         / data_obs_mist["T_mean_1"])**2
    )
)

# ============================================================
# RADIO SECUNDARIO — MIST
# ============================================================

data_obs_mist["R2_mean"] = (
    np.sqrt(1 - data_obs_mist["Depth_primary"])
    * data_obs_mist["R1_mean"]
)

data_obs_mist["R2_std"] = np.sqrt(
    (1 - data_obs_mist["Depth_primary"])
    * data_obs_mist["R1_std"]**2
    +
    (
        data_obs_mist["R1_mean"]
        * data_obs_mist["Depth_primary_err"]
        /
        (2 * np.sqrt(1 - data_obs_mist["Depth_primary"]))
    )**2
)

# ============================================================
# RADIO PRIMARIO — SOMERS+17
# ============================================================

data_obs_somers["R1_mean"] = (
    10**(data_obs_somers["lgL_mean_2"] / 2)
    * (T_sun / data_obs_somers["T_mean_2"])**2
)

data_obs_somers["R1_std"] = (
    data_obs_somers["R1_mean"]
    * np.sqrt(
        ((np.log(10) / 2) * data_obs_somers["lgL_std_2"])**2
        +
        (2 * data_obs_somers["T_std_2"]
         / data_obs_somers["T_mean_2"])**2
    )
)

# ============================================================
# RADIO SECUNDARIO — SOMERS+17
# ============================================================

data_obs_somers["R2_mean"] = (
    np.sqrt(1 - data_obs_somers["Depth_primary"])
    * data_obs_somers["R1_mean"]
)

data_obs_somers["R2_std"] = np.sqrt(
    (1 - data_obs_somers["Depth_primary"])
    * data_obs_somers["R1_std"]**2
    +
    (
        data_obs_somers["R1_mean"]
        * data_obs_somers["Depth_primary_err"]
        /
        (2 * np.sqrt(1 - data_obs_somers["Depth_primary"]))
    )**2
)


# ============================================================
# TICs A EXCLUIR
# ============================================================

tics_excluir = [
    11412481,
    264459857,
    264483228,
    138829403
]


# ============================================================
# FILTRAR DATOS OBSERVACIONALES
# ============================================================

data_obs_mist = data_obs_mist[
    ~data_obs_mist["TIC"].isin(tics_excluir)
].copy()

data_obs_somers = data_obs_somers[
    ~data_obs_somers["TIC"].isin(tics_excluir)
].copy()

display(
    data_obs_mist[
        [
            "TIC",
            "age_mean_1",
            "age_std_1",
            "R1_mean",
            "R1_std",
            "R2_mean",
            "R2_std"
        ]
    ].head()
)

display(
    data_obs_somers[
        [
            "TIC",
            "age_mean_2",
            "age_std_2",
            "R1_mean",
            "R1_std",
            "R2_mean",
            "R2_std"
        ]
    ].head()
)


In [ ]:

# Teoricos modelo mist
age_mist_teo = mist["star_age"] / 1e6 #convertida a millones de años
mass_mist_teo = mist['star_mass']
logL_mist_teo = mist['log_L'] 
logTeff_mist_teo = mist['log_Teff']
logR_mist_teo = mist['log_R']

# Teoricos modelo somers 17 
ageGyr_steo = somers17['Age(Gyr)']
ageMyr_steo = somers17['Age(Gyr)'] * 1000 #convertida a millones de años
logL_steo = somers17['log(L/Lsun)']
mass_steo = somers17['Mass']
logR_steo = somers17['log(R/Rsun)']


# DATOS TEORICOS DE LAS ISOCRONAS

In [ ]:
# vamos a construir un csv que contenga toda la información para el caso de somers y mist
#Suposicion clave: tenemos que los datos de la estrella primaria son los dominantes en el espectro
# por lo que el calculo inicial de massage nos da (R1,masa1,Teff1,Lum1), ademas con la curva de luz pudimos estimar la profundidad
#del tránsito de lo que obtenemos los datos de la estrella secundaria
# (Radio2, masa2, teff2, Lum2)

# Organizar los datos

In [ ]:
# ============================================================
# COMPROBAR UNIDADES DE EDAD
# ============================================================

print("Edad observacional MIST:")
print(data_obs_mist["age_mean_1"].describe())

print("\nEdad teórica MIST:")
print(mist["star_age"].describe())

print("\nEdad observacional Somers+17:")
print(data_obs_somers["age_mean_2"].describe())

print("\nEdad teórica Somers+17 [Myr]:")
print(ageMyr_steo.describe())

In [ ]:
# ============================================================
# LOG RADIO SECUNDARIO — MIST
# ============================================================

data_obs_mist["logR2_mean"] = np.log10(
    data_obs_mist["R2_mean"]
)

data_obs_mist["logR2_std"] = (
    data_obs_mist["R2_std"]
    /
    (data_obs_mist["R2_mean"] * np.log(10))
)

# ============================================================
# LOG RADIO SECUNDARIO — SOMERS+17
# ============================================================

data_obs_somers["logR2_mean"] = np.log10(
    data_obs_somers["R2_mean"]
)

data_obs_somers["logR2_std"] = (
    data_obs_somers["R2_std"]
    /
    (data_obs_somers["R2_mean"] * np.log(10))
)

# ============================================================
# EXCLUIR TIC SIN EDAD PARA EL AJUSTE — SOMERS+17
# ============================================================

data_obs_somers_fit = data_obs_somers[
    data_obs_somers["age_mean_2"].notna() &
    data_obs_somers["age_std_2"].notna()
].copy()

In [ ]:
# ============================================================
# VERIFICACIÓN DE LOS OBSERVABLES PARA EL AJUSTE
# ============================================================

display(
    data_obs_mist[
        [
            "TIC",
            "age_mean_1",
            "age_std_1",
            "R2_mean",
            "R2_std",
            "logR2_mean",
            "logR2_std"
        ]
    ]
)

display(
    data_obs_somers_fit[
        [
            "TIC",
            "age_mean_2",
            "age_std_2",
            "R2_mean",
            "R2_std",
            "logR2_mean",
            "logR2_std"
        ]
    ]
)

In [ ]:
# ============================================================
# OBSERVABLES PARA EL AJUSTE — MIST
# ============================================================

tic_mist = data_obs_mist["TIC"].to_numpy()

age_obs_mist = data_obs_mist["age_mean_1"].to_numpy()
age_err_mist = data_obs_mist["age_std_1"].to_numpy()

logR_obs_mist = data_obs_mist["logR2_mean"].to_numpy()
logR_err_mist = data_obs_mist["logR2_std"].to_numpy()

# ============================================================
# VARIABLES TEÓRICAS — MIST
# ============================================================

age_teo_mist = mist["star_age"].to_numpy() / 1e6
logR_teo_mist = mist["log_R"].to_numpy()

In [ ]:
# ============================================================
# PRUEBA DEL CHI² — PRIMERA FILA MIST
# ============================================================

i = 0

chi2_mist_test = (
    (age_obs_mist[i] - age_teo_mist)**2
    / age_err_mist[i]**2
    +
    (logR_obs_mist[i] - logR_teo_mist)**2
    / logR_err_mist[i]**2
)

print("Número de puntos del track:", len(chi2_mist_test))
print("χ² mínimo:", np.nanmin(chi2_mist_test))


best_index = np.nanargmin(chi2_mist_test)

print("Índice del mejor modelo:", best_index)
print("Edad:", age_teo_mist[best_index], "Myr")
print("logR:", logR_teo_mist[best_index])

# ============================================================
# MEJOR MODELO MIST — PRUEBA
# ============================================================

best_model_mist = mist.iloc[best_index]

display(best_model_mist)

# ============================================================
# PLOT DE PRUEBA — TIC 50897998
# PRIMARIA + SECUNDARIA
# ============================================================

# ------------------------------------------------------------
# Seleccionar la primera solución del TIC 50897998
# ------------------------------------------------------------

star_test = data_obs_mist[
    data_obs_mist["TIC"] == 50897998
].iloc[0]


# ------------------------------------------------------------
# Mejor modelo MIST encontrado para la secundaria
# ------------------------------------------------------------

best_model = mist.iloc[best_index]

M2_teo = best_model["star_mass"]
R2_teo = 10**best_model["log_R"]


# ------------------------------------------------------------
# Valores observacionales
# ------------------------------------------------------------

M1_obs = star_test["mass_mean_1"]
M1_err = star_test["mass_std_1"]

R1_obs = star_test["R1_mean"]
R1_err = star_test["R1_std"]

R2_obs = star_test["R2_mean"]
R2_err = star_test["R2_std"]



In [ ]:
# ============================================================
# PLOT LIMPIO — TIC 50897998
# MIST: AGE-RADIUS + HR DIAGRAM
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable


# ============================================================
# 1. SISTEMA
# ============================================================

TIC_target = 50897998

star = data_obs_mist[
    data_obs_mist["TIC"] == TIC_target
].iloc[0]


# ============================================================
# 2. OBSERVABLES PRIMARIA
# ============================================================

age_obs = star["age_mean_1"]
age_err = star["age_std_1"]

M1 = star["mass_mean_1"]
M1_err = star["mass_std_1"]

T1 = star["T_mean_1"]
T1_err = star["T_std_1"]

logL1 = star["lgL_mean_1"]
logL1_err = star["lgL_std_1"]

R1 = star["R1_mean"]
R1_err = star["R1_std"]


# ============================================================
# 3. OBSERVABLES SECUNDARIA
# ============================================================

R2 = star["R2_mean"]
R2_err = star["R2_std"]

logR2 = star["logR2_mean"]
logR2_err = star["logR2_std"]


# ============================================================
# 4. MEJOR MODELO MIST
# ============================================================

best_index = np.nanargmin(chi2_mist_test)

best = mist.iloc[best_index]

age_best = best["star_age"] / 1e6

M2 = best["star_mass"]

logR2_best = best["log_R"]
R2_best = 10**logR2_best

logL2 = best["log_L"]
logTeff2 = best["log_Teff"]

T2 = 10**logTeff2


# ============================================================
# 5. TEMPERATURA PRIMARIA EN LOG
# ============================================================

logTeff1 = np.log10(T1)


# ============================================================
# 6. IMPRIMIR RESULTADOS
# ============================================================

print("=" * 65)
print(f"TIC {TIC_target}")
print("=" * 65)

print(f"Edad observada:       {age_obs:.3f} ± {age_err:.3f} Myr")
print(f"Edad mejor MIST:      {age_best:.3f} Myr")

print()
print("PRIMARIA")
print(f"M1 = {M1:.3f} ± {M1_err:.3f} Msun")
print(f"T1 = {T1:.1f} ± {T1_err:.1f} K")
print(f"log L1 = {logL1:.3f} ± {logL1_err:.3f}")
print(f"R1 = {R1:.3f} ± {R1_err:.3f} Rsun")

print()
print("SECUNDARIA — MEJOR MIST")
print(f"M2 = {M2:.3f} Msun")
print(f"T2 = {T2:.1f} K")
print(f"log L2 = {logL2:.3f}")
print(f"R2 = {R2_best:.3f} Rsun")

print("=" * 65)


# ============================================================
# 7. PREPARAR MIST
# ============================================================

mist_plot = mist.copy()

mist_plot["age_Myr"] = (
    mist_plot["star_age"] / 1e6
)

mist_plot["R"] = (
    10**mist_plot["log_R"]
)


# ============================================================
# 8. MASAS DE REFERENCIA
# ============================================================

# Masa mínima y máxima del grid
mass_min = mist_plot["star_mass"].min()
mass_max = mist_plot["star_mass"].max()

norm = Normalize(
    vmin=mass_min,
    vmax=mass_max
)

cmap = plt.cm.viridis


# ============================================================
# 9. FIGURA
# ============================================================

fig, (ax1, ax2) = plt.subplots(
    1,
    2,
    figsize=(14, 6)
)


# ============================================================
# PANEL IZQUIERDO
# AGE VS RADIUS
# ============================================================

# ------------------------------------------------------------
# Seleccionar track cercano a M1
# ------------------------------------------------------------

idx_M1 = np.abs(
    mist_plot["star_mass"] - M1
).argmin()

M1_mist = mist_plot.iloc[idx_M1]["star_mass"]

track1 = mist_plot[
    np.isclose(
        mist_plot["star_mass"],
        M1_mist
    )
].copy()

track1 = track1.sort_values("age_Myr")


# ------------------------------------------------------------
# Seleccionar track cercano a M2
# ------------------------------------------------------------

idx_M2 = np.abs(
    mist_plot["star_mass"] - M2
).argmin()

M2_mist = mist_plot.iloc[idx_M2]["star_mass"]

track2 = mist_plot[
    np.isclose(
        mist_plot["star_mass"],
        M2_mist
    )
].copy()

track2 = track2.sort_values("age_Myr")


# ------------------------------------------------------------
# Dibujar tracks
# ------------------------------------------------------------

ax1.plot(
    track1["age_Myr"],
    track1["R"],
    lw=2,
    color="black",
    label=fr"MIST $M={M1_mist:.2f}\,M_\odot$"
)

ax1.plot(
    track2["age_Myr"],
    track2["R"],
    lw=2,
    color="tab:orange",
    label=fr"MIST $M={M2_mist:.2f}\,M_\odot$"
)


# ------------------------------------------------------------
# Primaria
# ------------------------------------------------------------

ax1.errorbar(
    age_obs,
    R1,
    xerr=age_err,
    yerr=R1_err,
    fmt="o",
    markersize=10,
    color="black",
    markeredgecolor="white",
    markeredgewidth=1.5,
    capsize=4,
    zorder=10,
    label="Primary"
)


# ------------------------------------------------------------
# Secundaria observada
# ------------------------------------------------------------

ax1.errorbar(
    age_obs,
    R2,
    xerr=age_err,
    yerr=R2_err,
    fmt="s",
    markersize=9,
    color="tab:orange",
    markeredgecolor="white",
    markeredgewidth=1.5,
    capsize=4,
    zorder=10,
    label="Secondary"
)


# ------------------------------------------------------------
# Mejor modelo
# ------------------------------------------------------------

ax1.scatter(
    age_best,
    R2_best,
    marker="*",
    s=220,
    color="gold",
    edgecolor="black",
    linewidth=1.2,
    zorder=20,
    label="Best-fit MIST"
)


# ------------------------------------------------------------
# Edad adoptada
# ------------------------------------------------------------

ax1.axvline(
    age_best,
    linestyle="--",
    color="gray",
    alpha=0.6
)


ax1.set_xlabel(
    "Age [Myr]",
    fontsize=13
)

ax1.set_ylabel(
    r"$R\,[R_\odot]$",
    fontsize=13
)

ax1.set_title(
    "Age–Radius",
    fontsize=14
)

ax1.grid(
    alpha=0.25
)

ax1.legend(
    fontsize=9
)


# ============================================================
# PANEL DERECHO
# HR DIAGRAM
# ============================================================

# ------------------------------------------------------------
# ISÓCRONA
# ------------------------------------------------------------

# Tolerancia de edad
age_tol = max(
    age_best * 0.005,
    0.01
)

iso = mist_plot[
    np.abs(
        mist_plot["age_Myr"] - age_best
    ) < age_tol
].copy()


print(
    f"Número de puntos de la isocrona: {len(iso)}"
)


# ------------------------------------------------------------
# Dibujar isocrona
# ------------------------------------------------------------

sc = ax2.scatter(
    iso["log_Teff"],
    iso["log_L"],
    c=iso["star_mass"],
    cmap=cmap,
    norm=norm,
    s=20,
    alpha=0.85,
    rasterized=True
)


# ------------------------------------------------------------
# Primaria
# ------------------------------------------------------------

ax2.scatter(
    logTeff1,
    logL1,
    marker="o",
    s=140,
    color="black",
    edgecolor="white",
    linewidth=1.5,
    zorder=20,
    label=fr"Primary ($M={M1:.2f}\,M_\odot$)"
)


# ------------------------------------------------------------
# Secundaria
# ------------------------------------------------------------

ax2.scatter(
    logTeff2,
    logL2,
    marker="s",
    s=130,
    color="tab:orange",
    edgecolor="white",
    linewidth=1.5,
    zorder=20,
    label=fr"Secondary ($M={M2:.2f}\,M_\odot$)"
)


# ------------------------------------------------------------
# Mejor modelo
# ------------------------------------------------------------

ax2.scatter(
    logTeff2,
    logL2,
    marker="*",
    s=240,
    color="gold",
    edgecolor="black",
    linewidth=1.2,
    zorder=25
)


# ------------------------------------------------------------
# Convención HR
# ------------------------------------------------------------

ax2.invert_xaxis()


ax2.set_xlabel(
    r"$\log(T_{\rm eff}/{\rm K})$",
    fontsize=13
)

ax2.set_ylabel(
    r"$\log(L/L_\odot)$",
    fontsize=13
)

ax2.set_title(
    f"HR Diagram — MIST ({age_best:.2f} Myr)",
    fontsize=14
)

ax2.grid(
    alpha=0.25
)

ax2.legend(
    fontsize=9
)


# ============================================================
# 10. COLORBAR
# ============================================================

sm = ScalarMappable(
    norm=norm,
    cmap=cmap
)

sm.set_array([])

cbar = fig.colorbar(
    sm,
    ax=[ax1, ax2],
    pad=0.025
)

cbar.set_label(
    r"Mass [$M_\odot$]",
    fontsize=13
)


# ============================================================
# 11. TÍTULO
# ============================================================

fig.suptitle(
    f"TIC {TIC_target} — MIST evolutionary models",
    fontsize=16
)

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# PLOT MEJORADO — TIC 50897998
# MIST: AGE-RADIUS + HR DIAGRAM
# ============================================================
#
# Requiere que ya existan en el notebook, igual que antes:
#   data_obs_mist   -> catálogo con los observables (fila por TIC)
#   mist            -> grid/tracks MIST (star_age, star_mass, log_R, log_L, log_Teff, ...)
#   chi2_mist_test  -> array de chi2 alineado con las filas de `mist`
#
# Cambios clave respecto al original:
#   1) La traza de una masa dada ya NO se selecciona con np.isclose(default),
#      que es demasiado estricta si la masa cambia levemente con la edad
#      (pérdida de masa / redondeo). Ahora se agrupa por la masa "inicial"
#      (edad mínima) de cada traza y se usa una tolerancia relativa.
#   2) Eje de edad en log en el panel izquierdo, apropiado para mostrar la
#      contracción pre-secuencia-principal sin que se vea como una espiga.
#   3) constrained_layout en vez de tight_layout -> ya no hay warning con
#      colorbar compartido entre los dos ejes.
#   4) Barras de error correctas (incluye log-Teff) en el diagrama HR.
#   5. Estética: tipografía, paleta, grosores y leyendas más limpias.
#
# NOTA IMPORTANTE (revisar antes de confiar en la figura):
#   Verifica que R2_mean / logR2_mean en `data_obs_mist` realmente
#   correspondan a la secundaria observada, y no al ajuste Somers (17%
#   spot coverage) de la MISMA estrella primaria. Según tu método de
#   trabajo, la secundaria se obtiene como R2 = R1 * sqrt(Depth_primary)
#   y luego se ubica en la isócrona (tú usas BHAC15 para eso, no MIST).
#   Si aquí quieres seguir usando MIST para la secundaria, asegúrate de
#   que R2/logR2 vengan de esa fórmula y no directo de columnas "_2".
# ============================================================

# ------------------------------------------------------------
# Estilo general (tipografía y limpieza)
# ------------------------------------------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.linewidth": 1.1,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "legend.frameon": False,
    "mathtext.fontset": "cm",
})

CMAP = plt.cm.cividis  # paleta perceptualmente uniforme y colorblind-safe

# ============================================================
# 1. SISTEMA Y OBSERVABLES
# ============================================================

TIC_target = 50897998
star = data_obs_mist[data_obs_mist["TIC"] == TIC_target].iloc[0]

age_obs, age_err = star["age_mean_1"], star["age_std_1"]

M1, M1_err = star["mass_mean_1"], star["mass_std_1"]
T1, T1_err = star["T_mean_1"], star["T_std_1"]
logL1, logL1_err = star["lgL_mean_1"], star["lgL_std_1"]
R1, R1_err = star["R1_mean"], star["R1_std"]

R2, R2_err = star["R2_mean"], star["R2_std"]
logR2_obs, logR2_obs_err = star["logR2_mean"], star["logR2_std"]

logTeff1 = np.log10(T1)
# propagación de error a log10(T1): dlogT = dT / (T * ln10)
logTeff1_err = T1_err / (T1 * np.log(10))

# ============================================================
# 2. MEJOR MODELO MIST
# ============================================================

best_index = np.nanargmin(chi2_mist_test)
best = mist.iloc[best_index]

age_best = best["star_age"] / 1e6
M2 = best["star_mass"]
logR2_best = best["log_R"]
R2_best = 10 ** logR2_best
logL2 = best["log_L"]
logTeff2 = best["log_Teff"]
T2 = 10 ** logTeff2

# ============================================================
# 3. PREPARAR GRID MIST
# ============================================================

mist_plot = mist.copy()
mist_plot["age_Myr"] = mist_plot["star_age"] / 1e6
mist_plot["R"] = 10 ** mist_plot["log_R"]

mass_col = "initial_mass" if "initial_mass" in mist_plot.columns else "star_mass"

mass_min, mass_max = mist_plot["star_mass"].min(), mist_plot["star_mass"].max()
norm = Normalize(vmin=mass_min, vmax=mass_max)


def get_track(df, target_mass, mass_col=mass_col, tol_frac=0.03):
    """Extrae la traza evolutiva completa (todas las edades) cuya masa es
    más cercana a target_mass. Robusto a que la masa cambie levemente con
    la edad: agrupa por la masa mínima-edad de cada traza y usa tolerancia
    relativa en vez de igualdad flotante exacta."""
    ref = (
        df.sort_values("age_Myr")
        .groupby(mass_col, as_index=False)
        .first()
    )
    closest_val = ref.loc[(ref[mass_col] - target_mass).abs().idxmin(), mass_col]
    track = df[np.isclose(df[mass_col], closest_val, rtol=tol_frac)]
    return track.sort_values("age_Myr"), closest_val


track1, M1_mist = get_track(mist_plot, M1)
track2, M2_mist = get_track(mist_plot, M2)

print("=" * 65)
print(f"TIC {TIC_target}")
print("=" * 65)
print(f"Edad observada:   {age_obs:.3f} +/- {age_err:.3f} Myr")
print(f"Edad mejor MIST:  {age_best:.3f} Myr")
print(f"Puntos en traza 1 (M~{M1_mist:.2f}): {len(track1)}")
print(f"Puntos en traza 2 (M~{M2_mist:.2f}): {len(track2)}")
print("=" * 65)

# ============================================================
# 4. FIGURA
# ============================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.8), layout="constrained")

# ------------------------------------------------------------
# PANEL IZQUIERDO — Age vs Radius (eje de edad en log)
# ------------------------------------------------------------

ax1.plot(track1["age_Myr"], track1["R"], lw=2.2, color="black",
          label=fr"MIST $M={M1_mist:.2f}\,M_\odot$")
ax1.plot(track2["age_Myr"], track2["R"], lw=2.2, color="tab:orange",
          label=fr"MIST $M={M2_mist:.2f}\,M_\odot$")

ax1.errorbar(age_obs, R1, xerr=age_err, yerr=R1_err, fmt="o", markersize=9,
              color="black", markeredgecolor="white", markeredgewidth=1.4,
              elinewidth=1.2, capsize=3, alpha=0.9, zorder=10, label="Primaria")

ax1.errorbar(age_obs, R2, xerr=age_err, yerr=R2_err, fmt="s", markersize=8,
              color="tab:orange", markeredgecolor="white", markeredgewidth=1.4,
              elinewidth=1.2, capsize=3, alpha=0.9, zorder=10, label="Secundaria")

ax1.scatter(age_best, R2_best, marker="*", s=260, color="gold",
             edgecolor="black", linewidth=1.1, zorder=20, label="Mejor ajuste MIST")

ax1.axvline(age_best, ls="--", color="gray", alpha=0.6, lw=1)

ax1.set_xscale("log")
ax1.xaxis.set_major_formatter(ScalarFormatter())
ax1.set_xlabel("Edad [Myr]", fontsize=13)
ax1.set_ylabel(r"$R\,[R_\odot]$", fontsize=13)
ax1.set_title("Edad-Radio", fontsize=14)
ax1.grid(alpha=0.2, which="both")
ax1.legend(fontsize=9, loc="upper right")

# ------------------------------------------------------------
# PANEL DERECHO — Diagrama HR
# ------------------------------------------------------------

age_tol = max(age_best * 0.01, 0.05)
iso = mist_plot[np.abs(mist_plot["age_Myr"] - age_best) < age_tol].copy()
iso = iso.sort_values("star_mass")

sc = ax2.scatter(iso["log_Teff"], iso["log_L"], c=iso["star_mass"], cmap=CMAP,
                   norm=norm, s=22, alpha=0.9, rasterized=True, zorder=5)

ax2.errorbar(logTeff1, logL1, xerr=logTeff1_err, yerr=logL1_err, fmt="o",
              markersize=11, color="black", markeredgecolor="white",
              markeredgewidth=1.4, elinewidth=1.2, capsize=3, zorder=20,
              label=fr"Primaria ($M={M1:.2f}\,M_\odot$)")

ax2.errorbar(logTeff2, logL2, xerr=None, yerr=None, fmt="s", markersize=10,
              color="tab:orange", markeredgecolor="white", markeredgewidth=1.4,
              zorder=20, label=fr"Secundaria ($M={M2:.2f}\,M_\odot$)")

ax2.scatter(logTeff2, logL2, marker="*", s=260, color="gold",
             edgecolor="black", linewidth=1.1, zorder=25)

ax2.invert_xaxis()
ax2.set_xlabel(r"$\log(T_{\rm eff}/{\rm K})$", fontsize=13)
ax2.set_ylabel(r"$\log(L/L_\odot)$", fontsize=13)
ax2.set_title(f"Diagrama HR — MIST ({age_best:.2f} Myr)", fontsize=14)
ax2.grid(alpha=0.2, which="both")
ax2.legend(fontsize=9, loc="lower left")

# ------------------------------------------------------------
# Colorbar compartida
# ------------------------------------------------------------

sm = ScalarMappable(norm=norm, cmap=CMAP)
sm.set_array([])
cbar = fig.colorbar(sm, ax=[ax1, ax2], pad=0.02, shrink=0.9)
cbar.set_label(r"Masa [$M_\odot$]", fontsize=12)

fig.suptitle(f"TIC {TIC_target} — Modelos evolutivos MIST", fontsize=16, y=1.03)

plt.savefig("TIC_50897998_MIST.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# OBSERVACIONALES — SOMERS17 (columnas "_2" del catálogo MASSAGE)
# ============================================================

TIC_target = 50897998

age_obs_somers17  = data_obs_somers_fit["age_mean_2"].values
age_err_somers17  = data_obs_somers_fit["age_std_2"].values
logR_obs_somers17 = data_obs_somers_fit["logR2_mean"].values
logR_err_somers17 = data_obs_somers_fit["logR2_std"].values

# ============================================================
# PRUEBA DEL CHI² — PRIMERA FILA SOMERS17
# ============================================================

i = 0

chi2_somers17_test = (
    (age_obs_somers17[i] - ageMyr_steo)**2
    / age_err_somers17[i]**2
    +
    (logR_obs_somers17[i] - logR_steo)**2
    / logR_err_somers17[i]**2
)

print("Número de puntos del track:", len(chi2_somers17_test))
print("χ² mínimo:", np.nanmin(chi2_somers17_test))

best_index = np.nanargmin(chi2_somers17_test)

print("Índice del mejor modelo:", best_index)
print("Edad:", ageMyr_steo[best_index], "Myr")
print("logR:", logR_steo[best_index])

# ============================================================
# MEJOR MODELO SOMERS17 — PRUEBA
# ============================================================

best_model_somers17 = somers17.iloc[best_index]

display(best_model_somers17)

display(
    data_obs_somers_fit[
        [
            "TIC",
            "age_mean_2",
            "age_std_2",
            "R2_mean",
            "R2_std",
            "logR2_mean",
            "logR2_std"
        ]
    ]
)

In [ ]:
# ============================================================
# 1. SISTEMA Y OBSERVABLES — ajuste con Somers17
# ============================================================

TIC_target = 72829000
star = data_obs_somers_fit[data_obs_somers_fit["TIC"] == TIC_target].iloc[0]

age_obs, age_err = star["age_mean_2"], star["age_std_2"]

M1, M1_err       = star["mass_mean_2"], star["mass_std_2"]
T1, T1_err       = star["T_mean_2"], star["T_std_2"]
logL1, logL1_err = star["lgL_mean_2"], star["lgL_std_2"]
R1, R1_err       = star["R1_mean"], star["R1_std"]

R2, R2_err = star["R2_mean"], star["R2_std"]
logR2_obs, logR2_obs_err = star["logR2_mean"], star["logR2_std"]

logTeff1 = np.log10(T1)
logTeff1_err = T1_err / (T1 * np.log(10))

# ============================================================
# 2. MEJOR MODELO SOMERS17
# ============================================================

best_index = np.nanargmin(chi2_somers17_test)
best = somers17.iloc[best_index]

age_best   = best["Age(Gyr)"] * 1000   # Myr
M2         = best["Mass"]
logR2_best = best["log(R/Rsun)"]
R2_best    = 10 ** logR2_best
logL2      = best["log(L/Lsun)"]

# Somers17 no siempre trae Teff tabulado. Si existe la columna, se usa;
# si no, se deriva por Stefan-Boltzmann en unidades solares:
# log(Teff/Teffsun) = [log(L/Lsun) - 2*log(R/Rsun)] / 4
TEFFSUN_LOG = np.log10(5772.0)
teff_col = next((c for c in ["log(Teff)", "logTeff", "Teff"] if c in somers17.columns), None)

if teff_col is not None:
    logTeff2 = best[teff_col] if "log" in teff_col.lower() else np.log10(best[teff_col])
else:
    logTeff2 = TEFFSUN_LOG + (logL2 - 2 * logR2_best) / 4

T2 = 10 ** logTeff2

# ============================================================
# 3. PREPARAR GRID SOMERS17
# ============================================================

somers17_plot = somers17.copy()
somers17_plot["age_Myr"] = somers17_plot["Age(Gyr)"] * 1000
somers17_plot["R"] = 10 ** somers17_plot["log(R/Rsun)"]

mass_col = "Mass"
mass_min, mass_max = somers17_plot["Mass"].min(), somers17_plot["Mass"].max()
norm = Normalize(vmin=mass_min, vmax=mass_max)


def get_track(df, target_mass, mass_col=mass_col, tol_frac=0.03):
    """Extrae la traza evolutiva completa (todas las edades) cuya masa es
    más cercana a target_mass."""
    ref = df.sort_values("age_Myr").groupby(mass_col, as_index=False).first()
    closest_val = ref.loc[(ref[mass_col] - target_mass).abs().idxmin(), mass_col]
    track = df[np.isclose(df[mass_col], closest_val, rtol=tol_frac)]
    return track.sort_values("age_Myr"), closest_val


track1, M1_somers = get_track(somers17_plot, M1)
track2, M2_somers = get_track(somers17_plot, M2)

print("=" * 65)
print(f"TIC {TIC_target}")
print("=" * 65)
print(f"Edad observada:      {age_obs:.3f} +/- {age_err:.3f} Myr")
print(f"Edad mejor Somers17: {age_best:.3f} Myr")
print(f"Puntos en traza 1 (M~{M1_somers:.2f}): {len(track1)}")
print(f"Puntos en traza 2 (M~{M2_somers:.2f}): {len(track2)}")
print("=" * 65)

# ============================================================
# 4. FIGURA
# ============================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.8), layout="constrained")

# ------------------------------------------------------------
# PANEL IZQUIERDO — Edad vs Radio
# ------------------------------------------------------------

ax1.plot(track1["age_Myr"], track1["R"], lw=2.2, color="black",
          label=fr"Somers17 $M={M1_somers:.2f}\,M_\odot$")
ax1.plot(track2["age_Myr"], track2["R"], lw=2.2, color="tab:orange",
          label=fr"Somers17 $M={M2_somers:.2f}\,M_\odot$")

ax1.errorbar(age_obs, R1, xerr=age_err, yerr=R1_err, fmt="o", markersize=9,
              color="black", markeredgecolor="white", markeredgewidth=1.4,
              elinewidth=1.2, capsize=3, alpha=0.9, zorder=10, label="Primaria")

ax1.errorbar(age_obs, R2, xerr=age_err, yerr=R2_err, fmt="s", markersize=8,
              color="tab:orange", markeredgecolor="white", markeredgewidth=1.4,
              elinewidth=1.2, capsize=3, alpha=0.9, zorder=10, label="Secundaria")

ax1.scatter(age_best, R2_best, marker="*", s=260, color="gold",
             edgecolor="black", linewidth=1.1, zorder=20, label="Mejor ajuste Somers17")

ax1.axvline(age_best, ls="--", color="gray", alpha=0.6, lw=1)

ax1.set_xscale("log")
ax1.xaxis.set_major_formatter(ScalarFormatter())
ax1.set_xlabel("Edad [Myr]", fontsize=13)
ax1.set_ylabel(r"$R\,[R_\odot]$", fontsize=13)
ax1.set_title("Edad-Radio", fontsize=14)
ax1.grid(alpha=0.2, which="both")
ax1.legend(fontsize=9, loc="upper right")

# ------------------------------------------------------------
# PANEL DERECHO — Diagrama HR
# ------------------------------------------------------------

age_tol = max(age_best * 0.01, 0.05)
iso = somers17_plot[np.abs(somers17_plot["age_Myr"] - age_best) < age_tol].copy()
iso = iso.sort_values("Mass")

if teff_col is not None:
    iso_logTeff = iso[teff_col] if "log" in teff_col.lower() else np.log10(iso[teff_col])
else:
    iso_logTeff = TEFFSUN_LOG + (iso["log(L/Lsun)"] - 2 * iso["log(R/Rsun)"]) / 4

sc = ax2.scatter(iso_logTeff, iso["log(L/Lsun)"], c=iso["Mass"], cmap=CMAP,
                   norm=norm, s=22, alpha=0.9, rasterized=True, zorder=5)

ax2.errorbar(logTeff1, logL1, xerr=logTeff1_err, yerr=logL1_err, fmt="o",
              markersize=11, color="black", markeredgecolor="white",
              markeredgewidth=1.4, elinewidth=1.2, capsize=3, zorder=20,
              label=fr"Primaria ($M={M1:.2f}\,M_\odot$)")

ax2.errorbar(logTeff2, logL2, xerr=None, yerr=None, fmt="s", markersize=10,
              color="tab:orange", markeredgecolor="white", markeredgewidth=1.4,
              zorder=20, label=fr"Secundaria ($M={M2:.2f}\,M_\odot$)")

ax2.scatter(logTeff2, logL2, marker="*", s=260, color="gold",
             edgecolor="black", linewidth=1.1, zorder=25)

ax2.invert_xaxis()
ax2.set_xlabel(r"$\log(T_{\rm eff}/{\rm K})$", fontsize=13)
ax2.set_ylabel(r"$\log(L/L_\odot)$", fontsize=13)
ax2.set_title(f"Diagrama HR — Somers17 ({age_best:.2f} Myr)", fontsize=14)
ax2.grid(alpha=0.2, which="both")
ax2.legend(fontsize=9, loc="lower left")

# ------------------------------------------------------------
# Colorbar compartida
# ------------------------------------------------------------

sm = ScalarMappable(norm=norm, cmap=CMAP)
sm.set_array([])
cbar = fig.colorbar(sm, ax=[ax1, ax2], pad=0.02, shrink=0.9)
cbar.set_label(r"Masa [$M_\odot$]", fontsize=12)

fig.suptitle(f"TIC {TIC_target} — Modelos evolutivos Somers17", fontsize=16, y=1.03)

plt.savefig(f"TIC_{TIC_target}_Somers17.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# RESUMEN DE PARÁMETROS — TIC SELECCIONADO
# MIST vs Somers17 · Primaria vs Secundaria
# ============================================================

TIC_target = 50897998  # <-- cambia aquí el TIC que quieras inspeccionar

def safe_get(row, col):
    """Devuelve el valor de una columna si existe, o 'N/D' si no está."""
    return row[col] if col in row.index else "N/D"

# ------------------------------------------------------------
# 1. Filas observadas en cada catálogo
# ------------------------------------------------------------

row_mist = data_obs_mist[data_obs_mist["TIC"] == TIC_target]
row_somers = data_obs_somers_fit[data_obs_somers_fit["TIC"] == TIC_target]

if row_mist.empty or row_somers.empty:
    raise ValueError(
        f"TIC {TIC_target} no encontrado en uno de los catálogos "
        f"(MIST: {'sí' if not row_mist.empty else 'NO'}, "
        f"Somers17: {'sí' if not row_somers.empty else 'NO'})"
    )

row_mist = row_mist.iloc[0]
row_somers = row_somers.iloc[0]

# ------------------------------------------------------------
# 2. Mejores modelos teóricos (ya calculados con el chi²)
# ------------------------------------------------------------

best_mist = mist.iloc[np.nanargmin(chi2_mist_test)]
best_somers = somers17.iloc[np.nanargmin(chi2_somers17_test)]

# ------------------------------------------------------------
# 3. Tabla resumen
# ------------------------------------------------------------

summary = pd.DataFrame({
    "MIST — Primaria (obs)": {
        "R [Rsun]":    safe_get(row_mist, "R1_mean"),
        "M [Msun]":    safe_get(row_mist, "mass_mean_1"),
        "Teff [K]":    safe_get(row_mist, "T_mean_1"),
        "log(L/Lsun)": safe_get(row_mist, "lgL_mean_1"),
    },
    "MIST — Secundaria (best-fit)": {
        "R [Rsun]":    10 ** best_mist["log_R"],
        "M [Msun]":    best_mist["star_mass"],
        "Teff [K]":    10 ** best_mist["log_Teff"],
        "log(L/Lsun)": best_mist["log_L"],
    },
    "Somers17 — Primaria (obs)": {
        "R [Rsun]":    safe_get(row_somers, "R1_mean"),
        "M [Msun]":    safe_get(row_somers, "mass_mean_2"),
        "Teff [K]":    safe_get(row_somers, "T_mean_2"),
        "log(L/Lsun)": safe_get(row_somers, "lgL_mean_2"),
    },
    "Somers17 — Secundaria (best-fit)": {
        "R [Rsun]":    10 ** best_somers["log(R/Rsun)"],
        "M [Msun]":    best_somers["Mass"],
        "Teff [K]":    "N/D (no tabulado; se puede derivar de L y R)",
        "log(L/Lsun)": best_somers["log(L/Lsun)"],
    },
}).T

print("=" * 70)
print(f"TIC {TIC_target} — Comparación MIST vs Somers17")
print("=" * 70)
display(summary)

# Interpolador

In [ ]:
# ============================================================
# PREPARAR MIST PARA INTERPOLACIÓN
# ============================================================

mist_interp = mist.copy()

mist_interp["age_Myr"] = mist_interp["star_age"] / 1e6

# Evitar valores no físicos
mist_interp = mist_interp[
    (mist_interp["age_Myr"] > 0) &
    np.isfinite(mist_interp["log_R"]) &
    np.isfinite(mist_interp["log_Teff"])
].copy()

# Coordenadas de interpolación
x_mist = np.log10(mist_interp["age_Myr"].to_numpy())
y_mist = mist_interp["log_R"].to_numpy()
z_mist = mist_interp["log_Teff"].to_numpy()

# Interpolador
interp_mist = LinearNDInterpolator(
    np.column_stack([x_mist, y_mist]),
    z_mist
)

In [ ]:
age_test = 20.0       # Myr
R_test = 0.8          # Rsun

logTeff_interp = interp_mist(
    np.log10(age_test),
    np.log10(R_test)
)

Teff_interp = 10**logTeff_interp

print("Teff interpolada:", Teff_interp, "K")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

sc = plt.scatter(
    x_mist,
    y_mist,
    c=z_mist,
    s=3,
    cmap="cividis",
    rasterized=True
)

plt.xlabel(r"$\log(t/{\rm Myr})$")
plt.ylabel(r"$\log(R/R_\odot)$")
plt.title("MIST: superficie de interpolación")

cbar = plt.colorbar(sc)
cbar.set_label(r"$\log(T_{\rm eff}/{\rm K})$")

plt.show()